# Feature Engineering for HAB + OISST Merge

This notebook loads `data/processed/merged/merged_hab_oisst.csv`, adds weekly lag features for selected predictor columns, creates a silicate-to-nitrate ratio, and optionally adds an `isHarmful` classification label.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Input: Katie's merged weekly HAB + OISST dataset.
INPUT_CSV = Path("data/processed/merged/merged_hab_oisst.csv")
# Output: a new CSV with the engineered features added.
OUTPUT_CSV = Path("data/processed/merged/merged_hab_oisst_features.csv")

# We compute lag features separately within each station.
GROUP_COL = "station"
# This column tells us the weekly order of observations.
TIME_COL = "week_start"
# These are the columns we want to create lag1 and lag2 features for.
LAG_COLUMNS = ["temp", "silicate", "nitrate", "avg_chloro"]

# Toggle this if the team wants a classification label in addition to pda.
ADD_CLASSIFICATION_LABEL = True
# A week is labeled harmful when pda is greater than this threshold.
CLASSIFICATION_THRESHOLD = 0.1

In [ ]:
# Load the merged dataset and parse week_start as a real datetime column.
df = pd.read_csv(INPUT_CSV, parse_dates=[TIME_COL])

# Check that the columns needed for feature engineering are present.
required_columns = {GROUP_COL, TIME_COL, "pda", *LAG_COLUMNS}
missing_columns = sorted(required_columns - set(df.columns))

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Sort by station and week so previous rows really mean previous weeks.
df = df.sort_values([GROUP_COL, TIME_COL]).reset_index(drop=True)
df[[TIME_COL, GROUP_COL, "pda", *LAG_COLUMNS]].head()

## What the lag features mean

- `temp_lag1` = previous week's `temp` for the same station
- `temp_lag2` = value from two weeks earlier for the same station
- Same pattern for `silicate`, `nitrate`, and `avg_chloro`
- If a previous value is missing, this notebook fills it with the current week's value for that same row

In [ ]:
# Group rows by station so lagged values come from the same pier/station only.
grouped = df.groupby(GROUP_COL, sort=False)

for column in LAG_COLUMNS:
    # Save the current week's values so we can use them as fallback fills.
    current_values = df[column]
    # lag1 = previous week's value for the same station.
    df[f"{column}_lag1"] = grouped[column].shift(1).fillna(current_values)
    # lag2 = value from two weeks earlier for the same station.
    df[f"{column}_lag2"] = grouped[column].shift(2).fillna(current_values)

# Replace nitrate=0 with NaN before division so we do not create infinite values.
safe_nitrate = df["nitrate"].replace(0, np.nan)
# This ratio may stay NaN when nitrate is 0, which is safer than inventing a fake value.
df["silicate_nitrate_ratio"] = df["silicate"] / safe_nitrate

if ADD_CLASSIFICATION_LABEL:
    # Convert the continuous pda value into a binary harmful/not harmful label.
    df["isHarmful"] = (df["pda"] > CLASSIFICATION_THRESHOLD).astype(int)

# Keep a list of the new engineered columns for quick inspection and summaries.
engineered_columns = [
    "temp_lag1", "temp_lag2",
    "silicate_lag1", "silicate_lag2",
    "nitrate_lag1", "nitrate_lag2",
    "avg_chloro_lag1", "avg_chloro_lag2",
    "silicate_nitrate_ratio",
]

if ADD_CLASSIFICATION_LABEL:
    engineered_columns.append("isHarmful")

df[[TIME_COL, GROUP_COL, *LAG_COLUMNS, *engineered_columns]].head(10)

In [ ]:
# Count how many missing values each engineered feature has.
summary = df[engineered_columns].isna().sum().rename("missing_values")
summary.to_frame()

In [ ]:
# Save the final feature-engineered dataset to a new CSV file.
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved engineered dataset to {OUTPUT_CSV.resolve()}")